## Script para cargar el modelo ResNet18 preentrenado en memoria.

In [1]:
import torch
from torchvision import models
from torchvision.models import ResNet18_Weights

In [4]:
device=None

if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(device)

print(f"Cargando ResNet18 en: {device}")

# Cargar modelo preentrenado con pesos de ImageNet
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Transferir a dispositivo
model = model.to(device)

print(f"✓ ResNet18 cargado exitosamente")
print(f"  Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Dispositivo: {device}")

# Mostrar arquitectura
print("\nArquitectura del modelo:")
print(model)

Cargando ResNet18 en: cpu
✓ ResNet18 cargado exitosamente
  Parámetros totales: 11,689,512
  Dispositivo: cpu

Arquitectura del modelo:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

## Evolució de les capes del FeatureExtractor

Input: `(batch_size, 3, 224, 224)`

| Paso | Capa | Operación | Dimensiones |
|------|------|-----------|-------------|
| 1 | `conv1` | Conv 7×7, stride=2, 64 filtros | `(B, 64, 112, 112)` |
| 2 | `bn1` | Batch normalization | `(B, 64, 112, 112)` |
| 3 | `relu` | ReLU activation | `(B, 64, 112, 112)` |
| 4 | `maxpool` | MaxPool 3×3, stride=2 | `(B, 64, 56, 56)` |
| 5 | `layer1` | ResNet block (sin stride) | `(B, 64, 56, 56)` |
| 6 | `layer2` | ResNet block, stride=2 | `(B, 128, 28, 28)` |
| 7 | `layer3` | ResNet block, stride=2 | `(B, 256, 14, 14)` |
| 8 | `layer4` | ResNet block, stride=2 | `(B, 512, 7, 7)` |
| 9 | `avgpool` | AdaptiveAvgPool2d(1) | `(B, 512, 1, 1)` |
| 10 | `flatten` | Aplanamiento spatial | `(B, 512)` |
| 11 | `intermediate` | Linear(512→256) + ReLU | `(B, 256)` |
| 12 | `classifier` | Linear(256→2) | `(B, 2)` |

Los ResNet Blocks són así:

```plaintext
Input: (B, 64, 56, 56)
    ↓
[Conv 3×3, stride=2, 128 canales] ← AQUÍ está el stride=2
    ↓
(B, 128, 28, 28)
    ↓
[Conv 3×3, stride=1, 128 canales] ← stride=1 mantiene tamaño
    ↓
(B, 128, 28, 28)
    ↓
Suma con input ajustado (mediante Conv 1×1, stride=2)
    ↓
Output: (B, 128, 28, 28)
```